# Notebook 4E : Test de Parallélisme Contrôlé (Fonctions Sleep)

**Objectif** : Vérifier que le DaskBackend parallélise correctement en utilisant des fonctions `sleep(X)` qui retournent un flux nul.

## Principe

- Créer N fonctions indépendantes qui dorment pendant X secondes
- Chaque fonction retourne un flux nul (ne modifie pas l'état)
- Temps séquentiel attendu : N × X
- Temps parallèle attendu (N workers) : X
- Speedup attendu : N×

Si le speedup est bon (~N×) → Dask fonctionne, le problème est dans les fonctions de calcul (GIL).  
Si le speedup est mauvais (~1×) → Le DaskBackend a un problème.

In [ ]:
import time
from pathlib import Path

import dask
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from seapopym.backend.dask import DaskBackend
from seapopym.blueprint.nodes import ComputeNode

print("✅ Imports réussis")

## Configuration

In [ ]:
CONFIG = {
    "n_tasks": 12,  # Nombre de tâches indépendantes
    "sleep_duration": 0.1,  # Durée de sleep par tâche (secondes)
    "grid_size": (100, 100),  # Grille légère
    "n_runs": 3,  # Nombre de répétitions pour moyenner
    "workers_list": [1, 2, 4, 6, 8, 12],
}

EXPECTED_SEQ_TIME = CONFIG["n_tasks"] * CONFIG["sleep_duration"]
print(f"Temps séquentiel attendu : {EXPECTED_SEQ_TIME:.2f}s")
print(f"Temps parallèle attendu (12 workers) : {CONFIG['sleep_duration']:.2f}s")
print(f"Speedup idéal max : {CONFIG['n_tasks']}×")

## Création des Fonctions Sleep

In [ ]:
def make_sleep_function(task_id: int, sleep_duration: float):
    """
    Crée une fonction sleep unique pour chaque tâche.
    Retourne un flux nul (ne modifie pas l'état).
    """

    def compute_sleep_tendency(state: xr.DataArray) -> dict[str, xr.DataArray]:
        time.sleep(sleep_duration)
        tendency = xr.zeros_like(state)
        return {f"tendency_{task_id}": tendency}

    compute_sleep_tendency.__name__ = f"sleep_task_{task_id}"
    return compute_sleep_tendency


# Créer les fonctions
sleep_functions = [
    make_sleep_function(i, CONFIG["sleep_duration"]) for i in range(CONFIG["n_tasks"])
]

print(f"✅ {len(sleep_functions)} fonctions sleep créées")

## Création de l'État Initial et des ComputeNodes

In [ ]:
# Créer un état minimal
ny, nx = CONFIG["grid_size"]
state = xr.Dataset(
    {
        f"tracer_{i}": xr.DataArray(
            np.ones((ny, nx)),
            dims=["y", "x"],
        )
        for i in range(CONFIG["n_tasks"])
    }
)

print(f"État créé : {list(state.data_vars)}")

# Créer les ComputeNodes
compute_nodes = []
for i, func in enumerate(sleep_functions):
    node = ComputeNode(
        name=f"sleep_task_{i}",
        func=func,
        input_mapping={"state": f"tracer_{i}"},
        output_mapping={f"tendency_{i}": f"tracer_{i}_tendency"},
    )
    compute_nodes.append(node)

# Format pour le backend : list of (group_name, tasks)
task_groups = [("sleep_group", compute_nodes)]

print(f"✅ {len(compute_nodes)} ComputeNodes créés")

## Test 1 : Exécution Séquentielle (Baseline)

In [ ]:
from seapopym.backend.core import execute_task_sequence


def run_sequential(task_groups, state, n_runs=3):
    """Exécute les tâches séquentiellement."""
    times = []
    all_tasks = [task for _, tasks in task_groups for task in tasks]

    for _ in range(n_runs):
        t_start = time.perf_counter()
        _ = execute_task_sequence(all_tasks, state, {})
        t_end = time.perf_counter()
        times.append(t_end - t_start)

    return np.mean(times), np.std(times)


t_seq_mean, t_seq_std = run_sequential(task_groups, state, CONFIG["n_runs"])
print(f"Temps séquentiel : {t_seq_mean:.3f}s ± {t_seq_std:.3f}s")
print(f"Temps attendu    : {EXPECTED_SEQ_TIME:.3f}s")
print(f"Écart            : {abs(t_seq_mean - EXPECTED_SEQ_TIME) / EXPECTED_SEQ_TIME * 100:.1f}%")

## Test 2 : Exécution Parallèle avec Dask ThreadPool

In [ ]:
def run_dask_parallel(task_groups, state, num_workers, n_runs=3):
    """Exécute les tâches avec Dask ThreadPool."""
    backend = DaskBackend()
    times = []

    for _ in range(n_runs):
        with dask.config.set(scheduler="threads", num_workers=num_workers):
            t_start = time.perf_counter()
            _ = backend.execute(task_groups, state)
            t_end = time.perf_counter()
        times.append(t_end - t_start)

    return np.mean(times), np.std(times)


# Test avec différents nombres de workers
results = []

print("\nTest de parallélisme Dask ThreadPool")
print("=" * 60)
print(f"{'Workers':<10} {'Temps (s)':<15} {'Speedup':<10} {'Efficacité':<10}")
print("-" * 60)

for n_workers in CONFIG["workers_list"]:
    t_mean, t_std = run_dask_parallel(task_groups, state, n_workers, CONFIG["n_runs"])
    speedup = t_seq_mean / t_mean
    efficiency = speedup / n_workers * 100

    results.append(
        {
            "workers": n_workers,
            "time_mean": t_mean,
            "time_std": t_std,
            "speedup": speedup,
            "efficiency": efficiency,
        }
    )

    print(f"{n_workers:<10} {t_mean:.3f} ± {t_std:.3f}   {speedup:.2f}×       {efficiency:.1f}%")

print("=" * 60)

## Visualisation des Résultats

In [ ]:
# Style publication
plt.rcParams.update(
    {
        "font.size": 10,
        "axes.labelsize": 10,
        "axes.titlesize": 11,
        "legend.fontsize": 9,
        "lines.linewidth": 1.5,
        "lines.markersize": 6,
    }
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

workers = [r["workers"] for r in results]
speedups = [r["speedup"] for r in results]
efficiencies = [r["efficiency"] for r in results]

# --- Speedup ---
ax1 = axes[0]
ax1.plot(workers, workers, "k--", label="Idéal (y=x)", alpha=0.7)
ax1.plot(workers, speedups, "o-", color="#0077BB", label="Mesuré")
ax1.set_xlabel("Nombre de Workers")
ax1.set_ylabel("Speedup")
ax1.set_title("Strong Scaling (Fonctions Sleep)")
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, max(workers) + 1)
ax1.set_ylim(0, max(workers) + 1)

# --- Efficacité ---
ax2 = axes[1]
ax2.axhline(100, color="k", linestyle="--", alpha=0.7, label="Idéal (100%)")
ax2.plot(workers, efficiencies, "s-", color="#EE7733", label="Mesuré")
ax2.set_xlabel("Nombre de Workers")
ax2.set_ylabel("Efficacité (%)")
ax2.set_title("Efficacité Parallèle")
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, max(workers) + 1)
ax2.set_ylim(0, 120)

plt.tight_layout()
plt.show()

# Sauvegarder
output_dir = Path("figures")
output_dir.mkdir(exist_ok=True)
fig.savefig(output_dir / "fig_04e_sleep_parallelism.png", dpi=300, bbox_inches="tight")
fig.savefig(output_dir / "fig_04e_sleep_parallelism.pdf", bbox_inches="tight")
print("✅ Figures sauvegardées")

## Analyse et Conclusions

In [ ]:
# Analyser les résultats
max_speedup = max(speedups)
max_workers = results[speedups.index(max_speedup)]["workers"]
avg_efficiency = np.mean(efficiencies)

print("\n" + "=" * 70)
print("ANALYSE DES RÉSULTATS")
print("=" * 70)

print(f"\nSpeedup maximum   : {max_speedup:.2f}× avec {max_workers} workers")
print(f"Efficacité moyenne: {avg_efficiency:.1f}%")

# Diagnostic
print("\n--- DIAGNOSTIC ---")

if max_speedup >= CONFIG["n_tasks"] * 0.8:
    print("✅ EXCELLENT : Le speedup est proche de l'idéal.")
    print("   → Dask ThreadPool fonctionne parfaitement.")
    print("   → Le problème de speedup dans 4B/4C vient des fonctions de calcul (GIL).")
    conclusion = "DASK_OK"
elif max_speedup >= CONFIG["n_tasks"] * 0.5:
    print("⚠️  MODÉRÉ : Le speedup est correct mais pas optimal.")
    print("   → Il y a probablement un overhead dans le DaskBackend.")
    conclusion = "PARTIAL"
else:
    print("❌ MAUVAIS : Le speedup est très faible.")
    print("   → Le DaskBackend ne parallélise pas correctement.")
    conclusion = "DASK_ISSUE"

# Calcul de l'overhead Dask
t_ideal_parallel = CONFIG["sleep_duration"]  # Temps idéal avec 12 workers
t_best_measured = min(
    [r["time_mean"] for r in results if r["workers"] >= CONFIG["n_tasks"]],
    default=results[-1]["time_mean"],
)
overhead = t_best_measured - t_ideal_parallel

print(f"\n--- OVERHEAD DASK ---")
print(f"Temps idéal (12 workers) : {t_ideal_parallel:.3f}s")
print(f"Temps mesuré             : {t_best_measured:.3f}s")
print(f"Overhead estimé          : {overhead:.3f}s ({overhead / t_best_measured * 100:.1f}%)")

## Génération du Résumé

In [ ]:
from datetime import datetime

summary = f"""====================================================================================================
NOTEBOOK 4E: TEST DE PARALLÉLISME CONTRÔLÉ (FONCTIONS SLEEP)
====================================================================================================

DATE: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

OBJECTIF:
----------------------------------------------------------------------------------------------------
Vérifier que le DaskBackend parallélise correctement en utilisant des fonctions
time.sleep() qui retournent un flux nul.

CONFIGURATION:
----------------------------------------------------------------------------------------------------
Nombre de tâches      : {CONFIG["n_tasks"]}
Durée sleep/tâche     : {CONFIG["sleep_duration"]}s
Grille                : {CONFIG["grid_size"][0]}×{CONFIG["grid_size"][1]}
Workers testés        : {CONFIG["workers_list"]}

RÉSULTATS:
----------------------------------------------------------------------------------------------------
Workers    Temps (s)       Speedup      Efficacité (%)
{"-" * 60}
"""

for r in results:
    summary += f"{r['workers']:<10} {r['time_mean']:.3f}           {r['speedup']:.2f}×        {r['efficiency']:.1f}\n"

summary += f"""
ANALYSE:
----------------------------------------------------------------------------------------------------
Speedup maximum       : {max_speedup:.2f}× avec {max_workers} workers
Efficacité moyenne    : {avg_efficiency:.1f}%
Overhead Dask estimé  : {overhead:.3f}s

CONCLUSION:
----------------------------------------------------------------------------------------------------
"""

if conclusion == "DASK_OK":
    summary += """✅ Dask ThreadPool fonctionne parfaitement.
   Le problème de speedup limité dans les notebooks 4B/4C provient des
   fonctions de calcul qui ne libèrent pas le GIL (guvectorize sans nogil)."""
elif conclusion == "PARTIAL":
    summary += """⚠️  Dask fonctionne mais avec un overhead non négligeable.
   Le DaskBackend pourrait être optimisé."""
else:
    summary += """❌ Le DaskBackend ne parallélise pas correctement.
   Investigation nécessaire."""

summary += f"""

FICHIERS GÉNÉRÉS:
----------------------------------------------------------------------------------------------------
- fig_04e_sleep_parallelism.pdf/png
- notebook_04e_sleep_summary.txt (ce fichier)

====================================================================================================
"""

# Sauvegarder le résumé
with open(output_dir / "notebook_04e_sleep_summary.txt", "w") as f:
    f.write(summary)

print(summary)

In [ ]:
# Test avec Distributed Client (processes)
from dask.distributed import Client


def run_dask_client(task_groups, state, num_workers, n_runs=3):
    """Exécute les tâches avec Dask Distributed Client."""
    backend = DaskBackend()
    times = []

    # Créer un client avec num_workers processes
    client = Client(n_workers=num_workers, threads_per_worker=1)

    try:
        for _ in range(n_runs):
            t_start = time.perf_counter()
            _ = backend.execute(task_groups, state)
            t_end = time.perf_counter()
            times.append(t_end - t_start)
    finally:
        client.close()

    return np.mean(times), np.std(times)


# Test
print("\nTest avec Distributed Client")
print("=" * 60)
for n_workers in [1, 2, 4, 8, 12]:
    t_mean, t_std = run_dask_client(task_groups, state, n_workers, 3)
    speedup = t_seq_mean / t_mean
    print(f"{n_workers} workers: {t_mean:.3f}s, speedup={speedup:.2f}×")


In [ ]:
# =============================================================================
# TEST RAPIDE : guvectorize vs @jit(nogil=True)
# =============================================================================
from numba import guvectorize, jit, float64, int32
import numpy as np
import xarray as xr


# --------------------------------------------------------------------------
# VERSION 1 : guvectorize (notre implémentation actuelle - NE libère PAS le GIL)
# --------------------------------------------------------------------------
@guvectorize(
    [(float64[:, :], float64[:, :])],
    "(y,x)->(y,x)",
    nopython=True,
    cache=True,
)
def compute_guvectorize(arr_in, arr_out):
    """Simule un calcul intensif avec guvectorize."""
    ny, nx = arr_in.shape
    for j in range(ny):
        for i in range(nx):
            # Calcul simulé (pas trop rapide pour voir la différence)
            total = 0.0
            for k in range(100):
                total += arr_in[j, i] * 0.01
            arr_out[j, i] = total


def task_guvectorize(data: xr.DataArray) -> dict:
    arr = data.values.copy()
    out = np.empty_like(arr)
    compute_guvectorize(arr, out)
    return {"tendency": xr.DataArray(out, dims=data.dims)}


# --------------------------------------------------------------------------
# VERSION 2 : @jit(nogil=True) - LIBÈRE le GIL
# --------------------------------------------------------------------------
@jit(nopython=True, nogil=True, cache=True)
def compute_jit_nogil(arr_in, arr_out):
    """Simule le même calcul mais libère le GIL."""
    ny, nx = arr_in.shape
    for j in range(ny):
        for i in range(nx):
            total = 0.0
            for k in range(100):
                total += arr_in[j, i] * 0.01
            arr_out[j, i] = total


def task_jit(data: xr.DataArray) -> dict:
    arr = data.values.copy()
    out = np.empty_like(arr)
    compute_jit_nogil(arr, out)
    return {"tendency": xr.DataArray(out, dims=data.dims)}


# --------------------------------------------------------------------------
# BENCHMARK
# --------------------------------------------------------------------------
# Créer données de test
test_data = xr.DataArray(np.random.rand(500, 500), dims=["y", "x"])
# Warmup (compilation JIT)
_ = task_guvectorize(test_data)
_ = task_jit(test_data)


def benchmark_function(func, data, n_tasks=12, n_workers=4, n_runs=3):
    """Benchmark une fonction avec Dask ThreadPool."""
    times = []

    for _ in range(n_runs):
        # Créer les tâches
        tasks = [dask.delayed(func)(data) for _ in range(n_tasks)]

        with dask.config.set(scheduler="threads", num_workers=n_workers):
            t_start = time.perf_counter()
            dask.compute(*tasks)
            t_end = time.perf_counter()
        times.append(t_end - t_start)

    return np.mean(times)


print("\n" + "=" * 60)
print("TEST : guvectorize vs @jit(nogil=True)")
print("=" * 60)
print(f"Config: 12 tâches indépendantes, grille 500×500, 100 itérations/cellule\n")
workers_list = [1, 2, 4, 8, 12]
print("Workers  | guvectorize  | @jit(nogil) | Ratio")
print("-" * 55)
for n_workers in workers_list:
    t_guv = benchmark_function(task_guvectorize, test_data, n_workers=n_workers)
    t_jit = benchmark_function(task_jit, test_data, n_workers=n_workers)
    ratio = t_guv / t_jit

    print(f"{n_workers:^8} | {t_guv:^12.3f} | {t_jit:^11.3f} | {ratio:.2f}×")
print("=" * 60)
print("\n✅ Si le ratio est ~1× → Le GIL n'est pas le problème")
print("✅ Si @jit(nogil) est beaucoup plus rapide → Le GIL EST le problème")
